<a href="https://colab.research.google.com/github/logonia/DAP/blob/main/Latest_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# COMPLETE UPDATED ANALYSIS SCRIPT
# - Addresses all supervisor comments
# - Fixes Oxford-IIIT Pets Dice evaluation
# - Saves all outputs as CSV
# ============================================================

import os
import random
import time
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, lr_scheduler
from torch.utils.data import DataLoader, random_split, Subset, Dataset
from torchvision import datasets, transforms

# ---------------------------
# Set up device & reproducibility
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(42)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# ---------------------------
# Helper functions
# ---------------------------
def squash(inputs, axis=-1):
    norm = torch.norm(inputs, p=2, dim=axis, keepdim=True)
    scale = (norm ** 2) / (1.0 + norm ** 2)
    return scale * inputs / (norm + 1e-8)

def to_onehot(y, num_classes):
    return torch.eye(num_classes, device=y.device)[y]

def caps_loss(y_true, y_pred, x, x_recon, lam_recon):
    margin = (y_true * torch.clamp(0.9 - y_pred, min=0.0)**2 +
              0.5 * (1.0 - y_true) * torch.clamp(y_pred - 0.1, min=0.0)**2)
    margin_loss = margin.sum(dim=1).mean()
    recon_loss = F.mse_loss(x_recon, x)
    return margin_loss + lam_recon * recon_loss

# ---------------------------
# Data loaders (CIFAR-10)
# ---------------------------
class ThresholdMaskTransform:
    def __init__(self, threshold):
        self.threshold = threshold
    def __call__(self, x):
        return x * (x > self.threshold).float()

class BackgroundNoiseTransform:
    def __init__(self, noise_std=0.25):
        self.noise_std = noise_std
    def __call__(self, x):
        return torch.clamp(x + torch.randn_like(x) * self.noise_std, 0.0, 1.0)

def load_cifar10(data_dir="./data", batch_size=128, val_size=5000, threshold=None):
    train_ops = [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(), transforms.ToTensor()]
    eval_ops = [transforms.ToTensor()]
    if threshold is not None:
        train_ops.append(ThresholdMaskTransform(threshold))
        eval_ops.append(ThresholdMaskTransform(threshold))

    train_tf = transforms.Compose(train_ops)
    eval_tf = transforms.Compose(eval_ops)

    full_train_aug = datasets.CIFAR10(data_dir, train=True, download=True, transform=train_tf)
    full_train_eval = datasets.CIFAR10(data_dir, train=True, download=False, transform=eval_tf)
    test_set = datasets.CIFAR10(data_dir, train=False, download=True, transform=eval_tf)

    total_len = len(full_train_aug)
    train_len = total_len - val_size
    train_subset, val_subset = random_split(range(total_len), [train_len, val_size],
                                            generator=torch.Generator().manual_seed(42))
    train_set = Subset(full_train_aug, train_subset.indices)
    val_set = Subset(full_train_eval, val_subset.indices)

    pin = torch.cuda.is_available()
    g = torch.Generator().manual_seed(42)
    train_loader = DataLoader(train_set, batch_size, shuffle=True, num_workers=2, pin_memory=pin,
                              worker_init_fn=seed_worker, generator=g)
    val_loader   = DataLoader(val_set, batch_size, shuffle=False, num_workers=2, pin_memory=pin,
                              worker_init_fn=seed_worker, generator=g)
    test_loader  = DataLoader(test_set, batch_size, shuffle=False, num_workers=2, pin_memory=pin,
                              worker_init_fn=seed_worker, generator=g)
    return train_loader, val_loader, test_loader

def load_cifar10_corrupted_test(data_dir="./data", batch_size=128, noise_std=0.25):
    transform = transforms.Compose([transforms.ToTensor(), BackgroundNoiseTransform(noise_std)])
    test_set = datasets.CIFAR10(data_dir, train=False, download=True, transform=transform)
    return DataLoader(test_set, batch_size, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available(),
                      worker_init_fn=seed_worker, generator=torch.Generator().manual_seed(42))

# ---------------------------
# Capsule components
# ---------------------------
class PrimaryCapsuleBase(nn.Module):
    def __init__(self, in_c=256, maps=32, dims=8):
        super().__init__()
        self.maps, self.dims = maps, dims
        self.conv = nn.Conv2d(in_c, maps*dims, 9, stride=2)
    def forward(self, x):
        out = self.conv(x)
        b, _, h, w = out.shape
        out = out.view(b, self.maps, self.dims, h, w).permute(0,1,3,4,2).contiguous()
        out = squash(out, axis=-1)
        return out.reshape(b, -1, self.dims), h, w

class PrimaryCapsuleUnsquashed(nn.Module):
    def __init__(self, in_c=256, maps=32, dims=8):
        super().__init__()
        self.maps, self.dims = maps, dims
        self.conv = nn.Conv2d(in_c, maps*dims, 9, stride=2)
    def forward(self, x):
        out = self.conv(x)
        b, _, h, w = out.shape
        out = out.view(b, self.maps, self.dims, h, w).permute(0,1,3,4,2).contiguous()
        raw_norms = torch.norm(out, dim=-1)
        return out, raw_norms, h, w

class DenseCapsule(nn.Module):
    def __init__(self, in_caps, out_caps, in_dims, out_dims, routings=3):
        super().__init__()
        self.in_caps = in_caps
        self.out_caps = out_caps
        self.routings = routings
        self.W = nn.Parameter(0.01 * torch.randn(out_caps, in_caps, out_dims, in_dims))
    def forward(self, x):
        u_hat = torch.einsum('bid,oijd->boij', x, self.W)
        b = torch.zeros(x.size(0), self.out_caps, self.in_caps, device=x.device)
        for i in range(self.routings):
            c = F.softmax(b, dim=1)
            s = (c.unsqueeze(-1) * u_hat).sum(dim=2)
            v = squash(s, axis=-1)
            if i < self.routings-1:
                b = b + (u_hat * v.unsqueeze(2)).sum(dim=-1)
        return v

class ImprovedCapFG(nn.Module):
    def __init__(self, in_maps=32, beta=0.2, entropy_weight=0.01, magnitude_weight=0.0, target_mean=0.3):
        super().__init__()
        self.beta = beta
        self.entropy_weight = entropy_weight
        self.magnitude_weight = magnitude_weight
        self.target_mean = target_mean
        self.local = nn.Sequential(
            nn.Conv2d(in_maps, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 1)
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(in_maps, 1)
        self.temperature = nn.Parameter(torch.tensor(1.0))
    def forward(self, raw_norms):
        logits = self.local(raw_norms)
        g = torch.sigmoid(self.fc(self.global_pool(raw_norms).flatten(1)))
        logits = logits * g[:, :, None, None]
        return torch.sigmoid(logits / self.temperature)
    def entropy_loss(self, mask):
        m = mask.view(mask.size(0), -1)
        entropy = -torch.mean(m * torch.log(m+1e-8) + (1-m)*torch.log(1-m+1e-8))
        return self.entropy_weight * entropy
    def magnitude_loss(self, mask):
        mean_act = mask.mean()
        return self.magnitude_weight * F.relu(self.target_mean - mean_act)

class CapsuleNetAblation(nn.Module):
    def __init__(self, shape=(3,32,32), classes=10, routings=3,
                 use_input_mask=False, use_capfg=False, mask_threshold=0.1,
                 beta=0.2, entropy_weight=0.01, magnitude_weight=0.0, target_mean=0.3):
        super().__init__()
        self.shape = shape
        self.classes = classes
        self.use_input_mask = use_input_mask
        self.use_capfg = use_capfg
        self.mask_threshold = mask_threshold

        self.conv1 = nn.Conv2d(shape[0], 256, kernel_size=9, stride=1, padding=0)
        self.relu = nn.ReLU(inplace=True)

        if use_capfg:
            self.primary = PrimaryCapsuleUnsquashed(256, 32, 8)
            self.mask_gen = ImprovedCapFG(32, beta=beta, entropy_weight=entropy_weight,
                                          magnitude_weight=magnitude_weight, target_mean=target_mean)
            self.num_caps_in = 32 * 8 * 8
        else:
            self.primary = PrimaryCapsuleBase(256, 32, 8)
            self.num_caps_in = 32 * 8 * 8

        self.digitcaps = DenseCapsule(self.num_caps_in, classes, 8, 16, routings)
        self.decoder = nn.Sequential(
            nn.Linear(16*classes, 512), nn.ReLU(inplace=True),
            nn.Linear(512, 1024), nn.ReLU(inplace=True),
            nn.Linear(1024, shape[0]*shape[1]*shape[2]), nn.Sigmoid()
        )

    def forward(self, x, y=None, return_mask=False):
        if self.use_input_mask:
            x = x * (x > self.mask_threshold).float()
        out = self.relu(self.conv1(x))
        if self.use_capfg:
            prim_unsq, raw_norms, h, w = self.primary(out)
            mask = self.mask_gen(raw_norms)
            mask_exp = mask.unsqueeze(-1)
            prim_masked = prim_unsq * mask_exp + prim_unsq * self.mask_gen.beta * (1 - mask_exp)
            prim_squashed = squash(prim_masked, axis=-1)
            prim_flat = prim_squashed.view(x.size(0), -1, 8)
        else:
            prim_flat, h, w = self.primary(out)
        out = self.digitcaps(prim_flat)
        length = out.norm(dim=-1)
        if y is None:
            idx = length.max(1)[1]
            y = torch.eye(self.classes, device=x.device)[idx]
        recon = self.decoder((out * y[:,:,None]).view(out.size(0), -1))
        recon = recon.view(-1, *self.shape)
        if return_mask and self.use_capfg:
            return length, recon, mask
        return length, recon

    def mask_extra_losses(self, mask):
        if self.use_capfg:
            return self.mask_gen.entropy_loss(mask) + self.mask_gen.magnitude_loss(mask)
        return torch.tensor(0.0, device=device)

# ---------------------------
# Training and evaluation
# ---------------------------
def evaluate_capsule(model, loader, lam_recon, num_classes, use_mask_loss=False):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            y_onehot = to_onehot(y, num_classes)
            if use_mask_loss and hasattr(model, 'mask_extra_losses'):
                y_pred, x_recon, mask = model(x, return_mask=True)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, lam_recon) + model.mask_extra_losses(mask)
            else:
                y_pred, x_recon = model(x)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, lam_recon)
            total_loss += loss.item() * x.size(0)
            correct += (y_pred.argmax(1) == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total

def train_one_run(model, train_loader, val_loader, config, use_mask_loss=False, model_name="model"):
    opt = Adam(model.parameters(), lr=config['lr'])
    sched = lr_scheduler.ExponentialLR(opt, gamma=config['lr_decay'])
    best_acc = -1.0
    best_epoch = 0
    best_state = None
    history = []

    for epoch in range(config['epochs']):
        model.train()
        train_loss_sum, train_total = 0.0, 0
        start = time.time()

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            y_onehot = to_onehot(y, config['classes'])
            opt.zero_grad()
            if use_mask_loss and hasattr(model, 'mask_extra_losses'):
                y_pred, x_recon, mask = model(x, y_onehot, return_mask=True)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon']) + model.mask_extra_losses(mask)
            else:
                y_pred, x_recon = model(x, y_onehot)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, config['lam_recon'])
            loss.backward()
            opt.step()
            train_loss_sum += loss.item() * x.size(0)
            train_total += x.size(0)

        sched.step()
        train_loss = train_loss_sum / train_total
        val_loss, val_acc = evaluate_capsule(model, val_loader, config['lam_recon'], config['classes'], use_mask_loss)
        epoch_time = time.time() - start

        if val_acc > best_acc:
            best_acc = val_acc
            best_epoch = epoch + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        history.append({
            'model': model_name,
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'epoch_time_sec': epoch_time,
        })
        print(f"  {model_name} | Epoch {epoch+1:02d}/{config['epochs']} | "
              f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
              f"val_acc={val_acc:.4f} | best={best_acc:.4f} @ epoch {best_epoch}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_acc, best_epoch, pd.DataFrame(history)

# ---------------------------
# Analysis functions
# ---------------------------
def get_primary_capsules_for_routing(model, x, apply_capfg=True):
    if model.use_input_mask:
        x_in = x * (x > model.mask_threshold).float()
    else:
        x_in = x
    out = model.relu(model.conv1(x_in))
    if model.use_capfg:
        prim_unsq, raw_norms, h, w = model.primary(out)
        if apply_capfg:
            mask = model.mask_gen(raw_norms)
            mask_exp = mask.unsqueeze(-1)
            prim_masked = prim_unsq * mask_exp + prim_unsq * model.mask_gen.beta * (1 - mask_exp)
            prim_squashed = squash(prim_masked, axis=-1)
        else:
            prim_squashed = squash(prim_unsq, axis=-1)
        prim_flat = prim_squashed.view(x.size(0), -1, 8)
    else:
        prim_flat, h, w = model.primary(out)
    return prim_flat

def compute_routing_metrics(model, loader, device, apply_capfg=True, max_batches=None):
    model.eval()
    all_weighted_agreements = []
    all_mean_dot_agreements = []
    all_entropies = []
    with torch.no_grad():
        for batch_idx, (x, y) in enumerate(loader):
            if max_batches is not None and batch_idx >= max_batches:
                break
            x = x.to(device)
            prim_flat = get_primary_capsules_for_routing(model, x, apply_capfg=apply_capfg)
            digitcaps = model.digitcaps
            u_hat = torch.einsum('bid,oijd->boij', prim_flat, digitcaps.W)
            b = torch.zeros(x.size(0), digitcaps.out_caps, digitcaps.in_caps, device=x.device)
            for i in range(digitcaps.routings):
                c = F.softmax(b, dim=1)
                s = (c.unsqueeze(-1) * u_hat).sum(dim=2)
                v = squash(s, axis=-1)
                dot_agreement = (u_hat * v.unsqueeze(2)).sum(dim=-1)
                if i < digitcaps.routings - 1:
                    b = b + dot_agreement
            c_final = F.softmax(b, dim=1)
            s_final = (c_final.unsqueeze(-1) * u_hat).sum(dim=2)
            v_final = squash(s_final, axis=-1)
            dot_final = (u_hat * v_final.unsqueeze(2)).sum(dim=-1)
            weighted_agreement = (c_final * dot_final).sum(dim=1).mean(dim=1)
            mean_dot_agreement = dot_final.mean(dim=(1,2))
            entropy = -(c_final * torch.log(c_final + 1e-8)).sum(dim=1).mean(dim=1)
            all_weighted_agreements.extend(weighted_agreement.cpu().numpy())
            all_mean_dot_agreements.extend(mean_dot_agreement.cpu().numpy())
            all_entropies.extend(entropy.cpu().numpy())
    return {
        'weighted_agreement_mean': float(np.mean(all_weighted_agreements)),
        'weighted_agreement_std': float(np.std(all_weighted_agreements)),
        'mean_dot_agreement_mean': float(np.mean(all_mean_dot_agreements)),
        'mean_dot_agreement_std': float(np.std(all_mean_dot_agreements)),
        'routing_entropy_mean': float(np.mean(all_entropies)),
        'routing_entropy_std': float(np.std(all_entropies)),
    }

def run_shortcut_diagnostic(baseline_model, full_model, test_loader, device, baseline_orig_acc, full_orig_acc):
    baseline_model.eval()
    full_model.eval()
    baseline_correct, full_correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            outer_mask = torch.ones_like(x)
            outer_mask[:, :, 8:24, 8:24] = 0.0
            noise = torch.randn_like(x) * 0.5
            x_shifted = torch.clamp(x * (1 - outer_mask) + noise * outer_mask, 0, 1)
            y_pred, _ = baseline_model(x_shifted)
            baseline_correct += (y_pred.argmax(1) == y).sum().item()
            y_pred, _ = full_model(x_shifted)
            full_correct += (y_pred.argmax(1) == y).sum().item()
            total += x.size(0)
    return {
        'baseline_shifted_acc': baseline_correct / total,
        'full_shifted_acc': full_correct / total,
        'baseline_drop': baseline_orig_acc - baseline_correct / total,
        'full_drop': full_orig_acc - full_correct / total,
    }

def evaluate_noise_robustness(models, loader_func, config, noise_levels=(0.0, 0.1, 0.2, 0.3)):
    rows = []
    for nl in noise_levels:
        if nl == 0.0:
            _, _, loader = load_cifar10(batch_size=config['batch_size'])
        else:
            loader = loader_func(batch_size=config['batch_size'], noise_std=nl)
        for model_name, model, use_mask_loss in models:
            _, acc = evaluate_capsule(model, loader, config['lam_recon'], config['classes'], use_mask_loss=use_mask_loss)
            rows.append({'noise_std': nl, 'model': model_name, 'accuracy': acc})
    df = pd.DataFrame(rows)
    wide = df.pivot(index='noise_std', columns='model', values='accuracy').reset_index()
    if 'Baseline' in wide.columns and 'Full_CapFG' in wide.columns:
        wide['Full_minus_Baseline'] = wide['Full_CapFG'] - wide['Baseline']
    return df, wide

def compute_per_class_metrics(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            y_pred, _ = model(x)
            all_preds.extend(y_pred.argmax(1).cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    precision, recall, f1, support = precision_recall_fscore_support(
        all_labels, all_preds, labels=range(10), average=None, zero_division=0
    )
    return precision, recall, f1, support

def measure_cost(model, input_shape, batch_size=128, warmup=20, repeats=100):
    model.eval()
    params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    flops = None
    try:
        from thop import profile
        dummy = torch.randn(1, *input_shape).to(device)
        flops, _ = profile(model, inputs=(dummy,), verbose=False)
    except Exception as e:
        print(f"FLOPs not computed for {model.__class__.__name__}: {e}")
    dummy = torch.randn(batch_size, *input_shape).to(device)
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(dummy)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.time()
        for _ in range(repeats):
            _ = model(dummy)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = time.time() - start
    ms_per_batch = elapsed / repeats * 1000
    ms_per_image = ms_per_batch / batch_size
    return {
        'parameters': params,
        'trainable_parameters': trainable_params,
        'parameters_M': params / 1e6,
        'FLOPs': flops,
        'FLOPs_M': flops / 1e6 if flops is not None else np.nan,
        'inference_ms_per_batch': ms_per_batch,
        'inference_ms_per_image': ms_per_image,
    }

def summarize_training_dynamics(history_df):
    rows = []
    for model_name, g in history_df.groupby('model'):
        best_val = g['val_acc'].max()
        best_epoch = int(g.loc[g['val_acc'].idxmax(), 'epoch'])
        threshold_95 = 0.95 * best_val
        epochs_to_95 = int(g[g['val_acc'] >= threshold_95]['epoch'].min())
        last5_var = float(g.tail(5)['val_acc'].var(ddof=1)) if len(g) >= 5 else np.nan
        rows.append({
            'model': model_name,
            'best_val_acc': best_val,
            'best_epoch': best_epoch,
            'epochs_to_95pct_best': epochs_to_95,
            'final_train_loss': float(g.iloc[-1]['train_loss']),
            'final_val_loss': float(g.iloc[-1]['val_loss']),
            'mean_epoch_time_sec': float(g['epoch_time_sec'].mean()),
            'val_acc_variance_last5': last5_var,
        })
    return pd.DataFrame(rows)

# ---------------------------
# Oxford-IIIT Pets dataset
# ---------------------------
class OxfordPetsClsDataset(Dataset):
    def __init__(self, split='trainval', img_size=32, return_mask=False):
        self.return_mask = return_mask
        target_types = ['category', 'segmentation'] if return_mask else 'category'
        self.ds = datasets.OxfordIIITPet(
            root='./data', split=split, target_types=target_types, download=True
        )
        self.image_tf = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
        ])
        self.mask_tf = transforms.Compose([
            transforms.Resize((img_size, img_size), interpolation=transforms.InterpolationMode.NEAREST),
            transforms.PILToTensor(),
        ])
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        if self.return_mask:
            img, target = self.ds[idx]
            label, seg = target
            x = self.image_tf(img)
            seg_t = self.mask_tf(seg).float().squeeze(0)
            fg = (seg_t != 2).float().unsqueeze(0)  # 1=foreground, 0=background
            return x, int(label), fg
        img, label = self.ds[idx]
        return self.image_tf(img), int(label)

def load_oxford_pets(batch_size=64, val_size=700, seed=42, img_size=32):
    trainval = OxfordPetsClsDataset(split='trainval', img_size=img_size, return_mask=False)
    test_cls = OxfordPetsClsDataset(split='test', img_size=img_size, return_mask=False)
    test_mask = OxfordPetsClsDataset(split='test', img_size=img_size, return_mask=True)

    train_len = len(trainval) - val_size
    g = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(trainval, [train_len, val_size], generator=g)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2,
                              worker_init_fn=seed_worker, generator=torch.Generator().manual_seed(seed))
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_cls, batch_size=batch_size, shuffle=False, num_workers=2)
    dice_loader = DataLoader(test_mask, batch_size=batch_size, shuffle=False, num_workers=2)
    return train_loader, val_loader, test_loader, dice_loader

def evaluate_capsule_loader3_safe(model, loader, lam_recon, num_classes, use_mask_loss=False):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch in loader:
            if len(batch) == 3:
                x, y, _ = batch
            else:
                x, y = batch
            x, y = x.to(device), y.to(device)
            y_onehot = to_onehot(y, num_classes)
            if use_mask_loss and hasattr(model, 'mask_extra_losses'):
                y_pred, x_recon, mask = model(x, return_mask=True)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, lam_recon) + model.mask_extra_losses(mask)
            else:
                y_pred, x_recon = model(x)
                loss = caps_loss(y_onehot, y_pred, x, x_recon, lam_recon)
            total_loss += loss.item() * x.size(0)
            correct += (y_pred.argmax(1) == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total

# ---------------------------
# CORRECTED Dice evaluation
# ---------------------------
def evaluate_capfg_dice(model, dice_loader, device, threshold=0.5, max_batches=None):
    if not hasattr(model, 'mask_gen') or model.mask_gen is None:
        return {'mean_dice': np.nan, 'std_dice': np.nan, 'n_images': 0}
    model.eval()
    dice_scores = []
    with torch.no_grad():
        for b, (x, y, true_mask) in enumerate(dice_loader):
            if max_batches is not None and b >= max_batches:
                break
            x = x.to(device)
            true_mask = true_mask.to(device)
            if true_mask.dim() == 3:
                true_mask = true_mask.unsqueeze(1)
            _, _, learned_mask = model(x, return_mask=True)
            if learned_mask.dim() == 3:
                learned_mask = learned_mask.unsqueeze(1)
            pred_mask = F.interpolate(learned_mask, size=true_mask.shape[-2:], mode='bilinear', align_corners=False)
            pred_bin = (pred_mask >= threshold).float()
            intersection = (pred_bin * true_mask).sum(dim=(1,2,3))
            union = pred_bin.sum(dim=(1,2,3)) + true_mask.sum(dim=(1,2,3))
            dice = (2 * intersection + 1e-8) / (union + 1e-8)
            dice_scores.extend(dice.cpu().numpy().tolist())
    return {
        'mean_dice': float(np.mean(dice_scores)) if dice_scores else np.nan,
        'std_dice': float(np.std(dice_scores, ddof=1)) if len(dice_scores) > 1 else np.nan,
        'n_images': len(dice_scores),
    }

# ---------------------------
# Ablation runner
# ---------------------------
def train_ablation_on_dataset(dataset_name, train_loader, val_loader, test_loader, config, experiment_specs,
                              classes, output_prefix, shape=(3,32,32), save_models=True):
    models = {}
    histories = []
    clean_rows = []
    old_classes = config['classes']
    config['classes'] = classes

    for spec in experiment_specs:
        print(f"\n--- [{dataset_name}] Training {spec['name']} for {config['epochs']} epochs ---")
        model = CapsuleNetAblation(
            shape=shape,
            classes=classes,
            use_input_mask=spec['use_input_mask'],
            use_capfg=spec['use_capfg'],
            mask_threshold=config['capfg_threshold'],
            beta=config['capfg_beta'],
            entropy_weight=config['capfg_entropy_weight'],
            magnitude_weight=config['capfg_magnitude_weight'],
            target_mean=config['capfg_target_mean'],
        ).to(device)

        best_val, best_epoch, hist = train_one_run(
            model, train_loader, val_loader, config,
            use_mask_loss=spec['use_mask_loss'], model_name=f"{dataset_name}_{spec['name']}"
        )
        if save_models:
            torch.save(model.state_dict(), f"{output_prefix}_{spec['name'].lower()}_best.pt")
        models[spec['name']] = model
        histories.append(hist)

        test_loss, test_acc = evaluate_capsule_loader3_safe(
            model, test_loader, config['lam_recon'], classes, spec['use_mask_loss']
        )
        clean_rows.append({
            'dataset': dataset_name,
            'model': spec['name'],
            'best_val_acc': best_val,
            'best_epoch': best_epoch,
            'test_loss': test_loss,
            'test_acc': test_acc,
        })

    config['classes'] = old_classes
    return models, pd.DataFrame(clean_rows), pd.concat(histories, ignore_index=True)

# ---------------------------
# MAIN
# ---------------------------
def main():
    config = {
        'epochs': 30,
        'batch_size': 128,
        'pets_batch_size': 64,
        'lr': 0.001,
        'lr_decay': 0.9,
        'lam_recon': 0.0005 * 3 * 32 * 32,
        'classes': 10,
        'capfg_threshold': 0.10,
        'capfg_beta': 0.20,
        'capfg_entropy_weight': 0.01,
        'capfg_magnitude_weight': 0.0,
        'capfg_target_mean': 0.30,
        'run_pets_ablation': True,
    }

    experiment_specs = [
        {'name': 'Baseline', 'use_input_mask': False, 'use_capfg': False, 'use_mask_loss': False},
        {'name': 'MaskOnly', 'use_input_mask': True, 'use_capfg': False, 'use_mask_loss': False},
        {'name': 'CapFGOnly', 'use_input_mask': False, 'use_capfg': True, 'use_mask_loss': True},
        {'name': 'Full_CapFG', 'use_input_mask': True, 'use_capfg': True, 'use_mask_loss': True},
    ]

    # --------------------------------------------------------------
    # CIFAR-10 ablation
    # --------------------------------------------------------------
    print("Loading CIFAR-10...")
    train_loader, val_loader, test_loader = load_cifar10(batch_size=config['batch_size'])

    cifar_models, clean_df, history_df = train_ablation_on_dataset(
        dataset_name='CIFAR10', train_loader=train_loader, val_loader=val_loader,
        test_loader=test_loader, config=config, experiment_specs=experiment_specs,
        classes=10, output_prefix='cifar10_30epoch_tuned', save_models=True
    )
    print("\n=== CIFAR-10 Ablation Results ===")
    print(clean_df)

    baseline_model = cifar_models['Baseline']
    full_model = cifar_models['Full_CapFG']
    baseline_test_acc = float(clean_df.loc[clean_df['model'] == 'Baseline', 'test_acc'].iloc[0])
    full_test_acc = float(clean_df.loc[clean_df['model'] == 'Full_CapFG', 'test_acc'].iloc[0])

    # Routing metrics
    print("\n=== Routing Entropy and Agreement ===")
    routing_rows = []
    for label, model, apply_gate in [
        ('Baseline', baseline_model, True),
        ('Full_before_gate', full_model, False),
        ('Full_after_gate', full_model, True),
    ]:
        rm = compute_routing_metrics(model, test_loader, device, apply_capfg=apply_gate)
        routing_rows.append({'dataset': 'CIFAR10', 'model': label, **rm})
    routing_df = pd.DataFrame(routing_rows)
    print(routing_df)

    # Shortcut diagnostic
    print("\n=== Shortcut Diagnostic ===")
    shortcut = run_shortcut_diagnostic(
        baseline_model, full_model, test_loader, device,
        baseline_orig_acc=baseline_test_acc, full_orig_acc=full_test_acc
    )
    shortcut_df = pd.DataFrame([{'dataset': 'CIFAR10', **shortcut}])
    print(shortcut_df)

    # Noise robustness
    print("\n=== Noise Robustness ===")
    model_tuples = [(s['name'], cifar_models[s['name']], s['use_mask_loss']) for s in experiment_specs]
    noise_long_df, noise_wide_df = evaluate_noise_robustness(
        model_tuples, load_cifar10_corrupted_test, config, noise_levels=(0.0, 0.1, 0.2, 0.3)
    )
    print(noise_wide_df)

    # Computational cost
    print("\n=== Computational Cost ===")
    cost_rows = []
    for spec in experiment_specs:
        cost_rows.append({'dataset': 'CIFAR10', 'model': spec['name'],
                          **measure_cost(cifar_models[spec['name']], (3,32,32), batch_size=config['batch_size'])})
    cost_df = pd.DataFrame(cost_rows)
    base_params = int(cost_df.loc[cost_df['model']=='Baseline','parameters'].iloc[0])
    base_time = float(cost_df.loc[cost_df['model']=='Baseline','inference_ms_per_image'].iloc[0])
    cost_df['added_parameters_vs_baseline'] = cost_df['parameters'] - base_params
    cost_df['added_parameters_M_vs_baseline'] = cost_df['added_parameters_vs_baseline'] / 1e6
    cost_df['inference_overhead_pct_vs_baseline'] = ((cost_df['inference_ms_per_image'] / base_time) - 1) * 100
    print(cost_df)

    # Per-class metrics
    print("\n=== Per-class Metrics ===")
    classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
    base_prec, base_rec, base_f1, base_sup = compute_per_class_metrics(baseline_model, test_loader, device)
    full_prec, full_rec, full_f1, full_sup = compute_per_class_metrics(full_model, test_loader, device)
    per_class_df = pd.DataFrame({
        'dataset': 'CIFAR10', 'class': classes, 'support': base_sup,
        'baseline_precision': base_prec, 'baseline_recall': base_rec, 'baseline_f1': base_f1,
        'full_precision': full_prec, 'full_recall': full_rec, 'full_f1': full_f1,
        'delta_f1': full_f1 - base_f1,
    })
    print(per_class_df)

    # Training dynamics
    print("\n=== Training Dynamics ===")
    training_summary_df = summarize_training_dynamics(history_df)
    print(training_summary_df)

    # --------------------------------------------------------------
    # Oxford-IIIT Pets ablation and Dice (fixed)
    # --------------------------------------------------------------
    pets_clean_df = pd.DataFrame()
    pets_dice_df = pd.DataFrame()
    if config['run_pets_ablation']:
        print("\nLoading Oxford-IIIT Pets...")
        pets_train, pets_val, pets_test, pets_dice_loader = load_oxford_pets(batch_size=config['pets_batch_size'])
        pets_config = config.copy()
        pets_config['batch_size'] = config['pets_batch_size']
        pets_config['classes'] = 37
        pets_config['lr'] = 0.0005  # lower lr for Pets
        pets_config['epochs'] = 50   # more epochs

        pets_models, pets_clean_df, pets_history_df = train_ablation_on_dataset(
            dataset_name='OxfordIIITPets', train_loader=pets_train, val_loader=pets_val,
            test_loader=pets_test, config=pets_config, experiment_specs=experiment_specs,
            classes=37, output_prefix='pets_30epoch_tuned', save_models=True
        )
        print("\n=== Oxford-IIIT Pets Classification Results ===")
        print(pets_clean_df)

        # Dice evaluation (fixed)
        dice_rows = []
        for spec in experiment_specs:
            if spec['use_capfg']:
                d = evaluate_capfg_dice(pets_models[spec['name']], pets_dice_loader, device, threshold=0.5)
                dice_rows.append({'dataset': 'OxfordIIITPets', 'model': spec['name'], **d})
            else:
                dice_rows.append({'dataset': 'OxfordIIITPets', 'model': spec['name'],
                                  'mean_dice': np.nan, 'std_dice': np.nan, 'n_images': 0})
        pets_dice_df = pd.DataFrame(dice_rows)
        print("\n=== Oxford-IIIT Pets Dice Scores (Learned Masks) ===")
        print(pets_dice_df)

    # --------------------------------------------------------------
    # Save all outputs
    # --------------------------------------------------------------
    clean_df.to_csv('cifar10_ablation_results.csv', index=False)
    routing_df.to_csv('cifar10_routing_metrics.csv', index=False)
    shortcut_df.to_csv('cifar10_shortcut_diagnostic.csv', index=False)
    noise_wide_df.to_csv('cifar10_noise_robustness.csv', index=False)
    cost_df.to_csv('cifar10_computational_cost.csv', index=False)
    per_class_df.to_csv('cifar10_per_class_metrics.csv', index=False)
    training_summary_df.to_csv('cifar10_training_dynamics.csv', index=False)

    if config['run_pets_ablation']:
        pets_clean_df.to_csv('pets_classification_results.csv', index=False)
        pets_dice_df.to_csv('pets_dice_scores.csv', index=False)

    # Compact summary
    summary = {
        'cifar_baseline_acc': baseline_test_acc,
        'cifar_full_acc': full_test_acc,
        'cifar_gain': full_test_acc - baseline_test_acc,
        'added_params_M': float(cost_df.loc[cost_df['model']=='Full_CapFG','added_parameters_M_vs_baseline'].iloc[0]),
        'inference_overhead_pct': float(cost_df.loc[cost_df['model']=='Full_CapFG','inference_overhead_pct_vs_baseline'].iloc[0]),
    }
    if config['run_pets_ablation'] and not pets_dice_df.empty:
        summary['pets_full_dice'] = float(pets_dice_df.loc[pets_dice_df['model']=='Full_CapFG','mean_dice'].iloc[0])
    pd.DataFrame([summary]).to_csv('summary_metrics.csv', index=False)

    print("\n✅ All analyses completed. Results saved as CSV files.")
    print("   - cifar10_ablation_results.csv")
    print("   - cifar10_routing_metrics.csv")
    print("   - cifar10_shortcut_diagnostic.csv")
    print("   - cifar10_noise_robustness.csv")
    print("   - cifar10_computational_cost.csv")
    print("   - cifar10_per_class_metrics.csv")
    print("   - cifar10_training_dynamics.csv")
    if config['run_pets_ablation']:
        print("   - pets_classification_results.csv")
        print("   - pets_dice_scores.csv")
    print("   - summary_metrics.csv")

if __name__ == '__main__':
    main()

Using device: cuda
Loading CIFAR-10...


100%|██████████| 170M/170M [22:16<00:00, 128kB/s]



--- [CIFAR10] Training Baseline for 30 epochs ---
  CIFAR10_Baseline | Epoch 01/30 | train_loss=0.5665 | val_loss=0.4757 | val_acc=0.4318 | best=0.4318 @ epoch 1
  CIFAR10_Baseline | Epoch 02/30 | train_loss=0.4729 | val_loss=0.4407 | val_acc=0.4794 | best=0.4794 @ epoch 2
  CIFAR10_Baseline | Epoch 03/30 | train_loss=0.4411 | val_loss=0.4124 | val_acc=0.5208 | best=0.5208 @ epoch 3
  CIFAR10_Baseline | Epoch 04/30 | train_loss=0.4181 | val_loss=0.3941 | val_acc=0.5452 | best=0.5452 @ epoch 4
  CIFAR10_Baseline | Epoch 05/30 | train_loss=0.3975 | val_loss=0.3812 | val_acc=0.5608 | best=0.5608 @ epoch 5
  CIFAR10_Baseline | Epoch 06/30 | train_loss=0.3829 | val_loss=0.3685 | val_acc=0.5782 | best=0.5782 @ epoch 6
  CIFAR10_Baseline | Epoch 07/30 | train_loss=0.3719 | val_loss=0.3594 | val_acc=0.5896 | best=0.5896 @ epoch 7
  CIFAR10_Baseline | Epoch 08/30 | train_loss=0.3614 | val_loss=0.3552 | val_acc=0.5958 | best=0.5958 @ epoch 8
  CIFAR10_Baseline | Epoch 09/30 | train_loss=0.3542 

100%|██████████| 792M/792M [00:03<00:00, 212MB/s]
100%|██████████| 19.2M/19.2M [00:00<00:00, 124MB/s] 



--- [OxfordIIITPets] Training Baseline for 50 epochs ---
  OxfordIIITPets_Baseline | Epoch 01/50 | train_loss=0.7418 | val_loss=0.7099 | val_acc=0.0171 | best=0.0171 @ epoch 1
  OxfordIIITPets_Baseline | Epoch 02/50 | train_loss=0.6964 | val_loss=0.7010 | val_acc=0.0686 | best=0.0686 @ epoch 2
  OxfordIIITPets_Baseline | Epoch 03/50 | train_loss=0.6790 | val_loss=0.6788 | val_acc=0.1229 | best=0.1229 @ epoch 3
  OxfordIIITPets_Baseline | Epoch 04/50 | train_loss=0.6634 | val_loss=0.6757 | val_acc=0.1229 | best=0.1229 @ epoch 3
  OxfordIIITPets_Baseline | Epoch 05/50 | train_loss=0.6449 | val_loss=0.6702 | val_acc=0.1286 | best=0.1286 @ epoch 5
  OxfordIIITPets_Baseline | Epoch 06/50 | train_loss=0.6192 | val_loss=0.6628 | val_acc=0.1429 | best=0.1429 @ epoch 6
  OxfordIIITPets_Baseline | Epoch 07/50 | train_loss=0.5833 | val_loss=0.6648 | val_acc=0.1143 | best=0.1429 @ epoch 6
  OxfordIIITPets_Baseline | Epoch 08/50 | train_loss=0.5474 | val_loss=0.6646 | val_acc=0.1371 | best=0.1429 